<a href="https://colab.research.google.com/github/rebmcph1-afk/nova-ds-ai-bootcamp-2026/blob/main/Day5_Nova.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 5 — AI Ethics, Human-in-the-Loop, and Real-World Pitfalls

## Learning Objectives
- Identify bias in data and models
- Understand explainability vs. black-box models
- Detect data drift
- Evaluate fairness across groups
- Use AI to critique model risks

# 📘 AI Bias + Human‑in‑the‑Loop Demo (California Housing Dataset)

We will use AI alone and AI with Human in the Loop to create two models using the CA housing data. In this dataset we will look at income and regional bias

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

sns.set(style="whitegrid", font_scale=1.2)

In [ ]:
#Load data
data = fetch_california_housing(as_frame=True)
df = data.frame
df.head()

In [ ]:
#EDA
df.describe()

In [ ]:
sns.scatterplot(data=df, x="Longitude", y="Latitude", hue="MedHouseVal", palette="viridis")
plt.title("California Housing Prices by Location")
plt.show()

In [5]:
#Train/Test Split
X = df.drop("MedHouseVal", axis=1)
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
#Train Baseline Model (AI Alone)
model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
mae

In [ ]:
#Feature Importance (Detecting Proxy Bias)
importances = pd.Series(model.feature_importances_, index=X.columns)
importances.sort_values().plot(kind="barh", figsize=(10,6))
plt.title("Feature Importance")
plt.show()

In [ ]:
#Bias Test: Error by Income Level
#Human‑in‑the‑loop checkpoint:
#Students interpret whether the model is unfair to lower‑income areas
test_df = X_test.copy()
test_df["true"] = y_test
test_df["pred"] = preds
test_df["error"] = abs(test_df["true"] - test_df["pred"])

test_df["income_bin"] = pd.qcut(test_df["MedInc"], q=4, labels=["Q1","Q2","Q3","Q4"])
test_df.groupby("income_bin")["error"].mean()

In [ ]:
#Bias Test: Geographic Error
test_df["lat_bin"] = pd.qcut(test_df["Latitude"], q=4, labels=["North","Mid-North","Mid-South","South"])
test_df.groupby("lat_bin")["error"].mean()


In [ ]:
#Residual Map (Visual Bias Detection)
#clusters of high error inland or rural
plt.figure(figsize=(10,8))
plt.scatter(test_df["Longitude"], test_df["Latitude"], c=test_df["error"], cmap="coolwarm", s=20)
plt.colorbar(label="Prediction Error")
plt.title("Geographic Residual Map")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

In [ ]:
#Human in the loop correction/remove bias features
X2 = df.drop(["MedHouseVal", "Latitude", "Longitude"], axis=1)
y2 = df["MedHouseVal"]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42
)

model2 = RandomForestRegressor(n_estimators=200, random_state=42)
model2.fit(X2_train, y2_train)

preds2 = model2.predict(X2_test)
mae2 = mean_absolute_error(y2_test, preds2)
mae2

In [ ]:
#Compare AI alone vs AI with Human oversight
print("Baseline MAE (AI alone):", mae)
print("Fairness-adjusted MAE (Human-in-the-loop):", mae2)

Baseline MAE (AI alone): 0.3268
This is the error of the original model, which used all features — including Latitude and Longitude, which act as proxies for wealth and location‑based socioeconomic status.

This model is:

More accurate

More biased

Because it relies heavily on geographic proxies.

Fairness-adjusted MAE (Human-in-the-loop): 0.4606
This is the error after humans intervened and removed the biased features (Latitude, Longitude).

This model is:

Less accurate

More fair

Because it no longer uses location as a socioeconomic shortcut.